# Feature Extraction 2

With the first feature extraction notebook becoming very large, this one was created to produce the final working extractor.

In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
import pretty_midi

import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config.paths import MSD_METADATA_DIR, PROCESSED_DATA_DIR, LMD_MATCHED_DIR, NOTEBOOK_DATASETS_DIR

## Load Dataset

In [2]:
metadata_df = pd.read_csv(f"{PROCESSED_DATA_DIR}/metadata_index.csv")

In [3]:
def perform_extraction(metadata_df: pd.DataFrame, extraction_func: callable, limit: int = 10) -> pd.DataFrame:

    """Extract features using the provided function for MIDI files listed in the metadata DataFrame. 
    Args:
        metadata_df (pd.DataFrame): DataFrame containing metadata about the MIDI files, including paths.
        extraction_func (callable): Function that takes a MIDI file path and returns a dictionary of extracted features.
        limit (int, optional): Maximum number of rows to process for testing. Defaults to 10."""

    rows = []

    for idx, row in tqdm(metadata_df.iterrows(), total=metadata_df.shape[0]):

        if idx < limit:  # Process only the first `limit` rows for testing
            features = extraction_func(row["midi_path"])
        
            if features is not None:
                features["track_id"] = row["track_id"]
                features["artist"] = row["artist_name"]
                features["title"] = row["title"]
                rows.append(features)

    return pd.DataFrame(rows)

## Extract Features

### Extract Preliminary Features

In [4]:
def load_song_data(midi_path: str) -> dict:
    """Load and preprocess MIDI files from a given directory."""

    midi_path = Path(midi_path)
    midi_files = list(midi_path.glob("*.mid"))

    if not midi_files:
        return None

    all_notes = []

    low_notes = []
    mid_notes = []
    high_notes = []

    drum_notes = []

    instrument_programs = []
    instrument_counts = []

    LOW_MAX = 48
    MID_MAX = 72

    for file in midi_files:

        try:
            midi = pretty_midi.PrettyMIDI(str(file))
        except Exception:
            continue

        instrument_counts.append(len(midi.instruments))

        for instrument in midi.instruments:

            instrument_programs.append(instrument.program)

            target = drum_notes if instrument.is_drum else None

            for note in instrument.notes:

                note_tuple = (
                    note.start,
                    note.end,
                    note.pitch,
                    note.velocity
                )

                if instrument.is_drum:
                    drum_notes.append(note_tuple)
                    continue

                all_notes.append(note_tuple)

                if note.pitch < LOW_MAX:
                    low_notes.append(note_tuple)

                elif note.pitch < MID_MAX:
                    mid_notes.append(note_tuple)

                else:
                    high_notes.append(note_tuple)

    if len(all_notes) < 5:
        return None

    # --------------------------------------------------
    # Sort everything by onset
    # --------------------------------------------------

    all_notes.sort(key=lambda x: x[0])

    low_notes.sort(key=lambda x: x[0])
    mid_notes.sort(key=lambda x: x[0])
    high_notes.sort(key=lambda x: x[0])

    drum_notes.sort(key=lambda x: x[0])

    # --------------------------------------------------
    # Melody extraction
    # Highest note at each onset
    # --------------------------------------------------

    by_time = defaultdict(list)

    for start, end, pitch, velocity in all_notes:
        by_time[start].append(
            (pitch, end, velocity)
        )

    melody_notes = []

    for start, notes in by_time.items():

        pitch, end, velocity = max(
            notes,
            key=lambda x: x[0]
        )

        melody_notes.append(
            (start, end, pitch, velocity)
        )

    melody_notes.sort(key=lambda x: x[0])

    # --------------------------------------------------
    # Helper
    # --------------------------------------------------

    def unpack(notes):

        if len(notes) == 0:
            return {
                "starts": np.array([]),
                "ends": np.array([]),
                "pitches": np.array([]),
                "velocities": np.array([]),
                "durations": np.array([])
            }

        starts = np.array([n[0] for n in notes])
        ends = np.array([n[1] for n in notes])

        return {
            "starts": starts,
            "ends": ends,
            "pitches": np.array([n[2] for n in notes]),
            "velocities": np.array([n[3] for n in notes]),
            "durations": ends - starts
        }

    return {

        # --------------------------------
        # Metadata
        # --------------------------------

        "path": str(midi_path),
        "num_files": len(midi_files),

        "instrument_counts": instrument_counts,
        "instrument_programs": instrument_programs,

        # --------------------------------
        # Raw note collections
        # --------------------------------

        "notes": all_notes,
        "low_notes": low_notes,
        "mid_notes": mid_notes,
        "high_notes": high_notes,

        "drum_notes": drum_notes,
        "melody_notes": melody_notes,

        # --------------------------------
        # Global arrays
        # --------------------------------

        **unpack(all_notes),

        # --------------------------------
        # Register splits
        # --------------------------------

        "low": unpack(low_notes),
        "mid": unpack(mid_notes),
        "high": unpack(high_notes),

        # --------------------------------
        # Drum features
        # --------------------------------

        "drums": unpack(drum_notes),

        # --------------------------------
        # Melody features
        # --------------------------------

        "melody": unpack(melody_notes)
    }

### Basic Features

In [5]:
def extract_basic_features(song):

    pitches = song["pitches"]
    durations = song["durations"]
    velocities = song["velocities"]

    return {

        "note_count": len(pitches),

        "mean_pitch": np.mean(pitches),
        "std_pitch": np.std(pitches),
        "pitch_range": np.max(pitches) - np.min(pitches),

        "mean_duration": np.mean(durations),
        "std_duration": np.std(durations),

        "mean_velocity": np.mean(velocities),
        "std_velocity": np.std(velocities),

        "num_files": song["num_files"],
        "mean_instruments": np.mean(song["instrument_counts"])
    }

### Extract Chroma and Key

In [6]:
def estimate_key(chroma):
    """
    Simple Krumhansl-Schmuckler-style correlation key estimate.
    chroma: 12D normalized vector
    """

    major_profile = np.array([6.35,2.23,3.48,2.33,4.38,4.09,
                              2.52,5.19,2.39,3.66,2.29,2.88])

    minor_profile = np.array([6.33,2.68,3.52,5.38,2.60,3.53,
                              2.54,4.75,3.98,2.69,3.34,3.17])

    best_score = -1
    best_key = None

    keys = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]

    for i in range(12):
        rotated = np.roll(chroma, i)

        major_score = np.corrcoef(rotated, major_profile)[0,1]
        minor_score = np.corrcoef(rotated, minor_profile)[0,1]

        if major_score > best_score:
            best_score = major_score
            best_key = keys[i] + " major"

        if minor_score > best_score:
            best_score = minor_score
            best_key = keys[i] + " minor"

    return best_key, best_score

def extract_chroma_features(song):

    def normalise(x):
        return x / (x.sum() + 1e-9)

    def chroma_from_pitches(pitches):

        if len(pitches) == 0:
            return np.zeros(12)

        return normalise(
            np.bincount(
                pitches.astype(int) % 12,
                minlength=12
            )
        )

    low_chroma = chroma_from_pitches(
        song["low"]["pitches"]
    )

    mid_chroma = chroma_from_pitches(
        song["mid"]["pitches"]
    )

    high_chroma = chroma_from_pitches(
        song["high"]["pitches"]
    )

    total_chroma = normalise(
        low_chroma + mid_chroma + high_chroma
    )

    key, key_strength = estimate_key(
        total_chroma
    )

    return {

        "low_chroma": low_chroma,
        "mid_chroma": mid_chroma,
        "high_chroma": high_chroma,

        "total_chroma": total_chroma,

        "key": key,
        "key_strength": key_strength
    }

### Extract NGrams

In [7]:
from collections import Counter

def extract_ngram_features(song, n=3):

    def build_intervals(pitches):

        if len(pitches) < 2:
            return np.array([])

        return np.diff(pitches)

    def build_ngrams(intervals):

        if len(intervals) < n:
            return Counter()

        motifs = [
            tuple(intervals[i:i+n])
            for i in range(
                len(intervals) - n + 1
            )
        ]

        return Counter(motifs)

    melody_intervals = build_intervals(
        song["melody"]["pitches"]
    )

    low_intervals = build_intervals(
        song["low"]["pitches"]
    )

    mid_intervals = build_intervals(
        song["mid"]["pitches"]
    )

    high_intervals = build_intervals(
        song["high"]["pitches"]
    )

    return {

        "melody_intervals": melody_intervals,
        "melody_ngrams":
            build_ngrams(melody_intervals),

        "low_intervals": low_intervals,
        "low_ngrams":
            build_ngrams(low_intervals),

        "mid_intervals": mid_intervals,
        "mid_ngrams":
            build_ngrams(mid_intervals),

        "high_intervals": high_intervals,
        "high_ngrams":
            build_ngrams(high_intervals)
    }

### Extract Rhythm

In [8]:
def extract_rhythm_features(song):

    def histogram(values):

        if len(values) == 0:
            return np.zeros(10)

        hist = np.histogram(
            values,
            bins=np.logspace(-3, 1, 11)
        )[0]

        return hist / (hist.sum() + 1e-9)

    def density(starts, ends):

        if len(starts) < 2:
            return 0

        return len(starts) / (
            ends.max() - starts.min() + 1e-9
        )

    def iois(starts):

        if len(starts) < 2:
            return np.array([])

        return np.diff(np.sort(starts))

    return {

        "ioi_hist":
            histogram(iois(song["starts"])),

        "duration_hist":
            histogram(song["durations"]),

        "density":
            density(
                song["starts"],
                song["ends"]
            ),

        "drum_ioi_hist":
            histogram(
                iois(song["drums"]["starts"])
            ),

        "drum_density":
            density(
                song["drums"]["starts"],
                song["drums"]["ends"]
            ),

        "bass_ioi_hist":
            histogram(
                iois(song["low"]["starts"])
            ),

        "bass_density":
            density(
                song["low"]["starts"],
                song["low"]["ends"]
            ),

        "melody_ioi_hist":
            histogram(
                iois(song["melody"]["starts"])
            ),

        "melody_density":
            density(
                song["melody"]["starts"],
                song["melody"]["ends"]
            )
    }

### Structure

In [9]:
def extract_structure_features(
    song,
    n_segments=16
):

    starts = song["starts"]
    pitches = song["pitches"]

    if len(starts) < 10:
        return {}

    edges = np.linspace(
        starts.min(),
        starts.max(),
        n_segments + 1
    )

    segment_chroma = []

    for i in range(n_segments):

        mask = (
            (starts >= edges[i]) &
            (starts < edges[i+1])
        )

        seg_pitches = pitches[mask]

        if len(seg_pitches) == 0:
            chroma = np.zeros(12)

        else:
            chroma = np.bincount(
                seg_pitches % 12,
                minlength=12
            )

            chroma = (
                chroma /
                (chroma.sum() + 1e-9)
            )

        segment_chroma.append(chroma)

    segment_chroma = np.array(
        segment_chroma
    )

    chord_roots = np.argmax(
        segment_chroma,
        axis=1
    )

    harmonic_rhythm = np.mean(
        chord_roots[1:] != chord_roots[:-1]
    )

    segment_variation = np.mean([
        np.linalg.norm(
            segment_chroma[i]
            - segment_chroma[i+1]
        )
        for i in range(
            len(segment_chroma)-1
        )
    ])

    return {

        "segment_chroma":
            segment_chroma,

        "chord_progression":
            chord_roots,

        "harmonic_rhythm":
            harmonic_rhythm,

        "segment_variation":
            segment_variation
    }

### Entropy

In [10]:
from scipy.stats import entropy

def extract_entropy_features(song):

    def hist_entropy(values):

        if len(values) == 0:
            return 0

        hist, _ = np.histogram(
            values,
            bins=20
        )

        hist = (
            hist /
            (hist.sum() + 1e-9)
        )

        return entropy(hist)

    iois = np.diff(
        np.sort(song["starts"])
    )

    return {

        "pitch_entropy":
            hist_entropy(
                song["pitches"]
            ),

        "duration_entropy":
            hist_entropy(
                song["durations"]
            ),

        "ioi_entropy":
            hist_entropy(iois)
    }

### Perform Extraction

In [11]:
def extract_features(midi_path: str) -> dict:
    """Given the path to a directory containing MIDI files, extract a comprehensive set of musical features."""

    song = load_song_data(midi_path)

    if song is None:
        return None

    return {
        **extract_basic_features(song),
        **extract_chroma_features(song),
        **extract_ngram_features(song),
        **extract_rhythm_features(song),
        **extract_structure_features(song),
        **extract_entropy_features(song),
    }

In [12]:
all_features_10_df = perform_extraction(
    metadata_df,
    extract_features,
    limit=10
)

all_features_100_df = perform_extraction(
    metadata_df,
    extract_features,
    limit=100
)

all_features_1000_df = perform_extraction(
    metadata_df,
    extract_features,
    limit=1000
)

  0%|          | 0/31034 [00:00<?, ?it/s]

c:\Users\danie\.venvs\ds\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
  0%|          | 0/31034 [00:00<?, ?it/s]c:\Users\danie\.venvs\ds\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
  0%|          | 0/31034 [00:00<?, ?it/s]c:\Users\danie\.venvs\ds\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
100%|██████████| 31034/31034 [12:12<00:00, 42.37it/s]   


In [13]:
all_features_10_df.to_pickle(f"{NOTEBOOK_DATASETS_DIR}/all_features_limit_10.pkl")
all_features_100_df.to_pickle(f"{NOTEBOOK_DATASETS_DIR}/all_features_limit_100.pkl")
all_features_1000_df.to_pickle(f"{NOTEBOOK_DATASETS_DIR}/all_features_limit_1000.pkl")

In [14]:
all_features_1000_df.memory_usage(deep=True).sum() / 1024**2

np.float64(306.00381088256836)

## Produce Embeddings

Deal with n-grams using a n-gram vocabulary


In [15]:
features_df = all_features_10_df.copy()

In [16]:
features_df.head()

,note_count,mean_pitch,std_pitch,pitch_range,mean_duration,std_duration,mean_velocity,std_velocity,num_files,mean_instruments,...,segment_chroma,chord_progression,harmonic_rhythm,segment_variation,pitch_entropy,duration_entropy,ioi_entropy,track_id,artist,title
0,18928,61.048394,13.679155,69,0.307508,0.390621,91.760936,23.248512,4,13.750000,...,"[[0.22918454935602645, 0.0, 0.1107296137338105...","[0, 9, 9, 9, 9, 9, 0, 9, 9, 9, 0, 9, 7, 0, 0, 0]",0.466667,0.077834,2.632983,0.975266,0.720061,TRAAAGR128F425B14B,Cyndi Lauper,Into The Nightlife
1,20863,63.037387,11.449555,65,0.166876,0.223345,105.646743,15.864140,6,10.000000,...,"[[0.2593950504122279, 0.0, 0.0, 0.287809349220...","[3, 3, 5, 0, 0, 5, 5, 0, 5, 5, 2, 2, 2, 2, 7, 2]",0.533333,0.133852,2.390657,0.717437,0.412685,TRAAAZF12903CCCF6B,Matthew Wilder,Break My Stride
2,14141,58.134785,10.822386,72,0.838205,1.199980,82.439714,28.306069,5,11.000000,...,"[[0.13705583756321987, 0.0, 0.1793570219963124...","[7, 7, 7, 7, 2, 7, 7, 7, 7, 7, 9, 9, 5, 0, 0, 5]",0.400000,0.106609,2.328768,0.792585,0.765635,TRAABVM128F92CA9DC,Tesla,Caught In A Dream
3,16762,63.042656,10.898794,74,0.632178,0.750975,88.762320,24.536796,5,9.600000,...,"[[0.2153846153843787, 0.0021978021977997826, 0...","[0, 0, 7, 7, 9, 7, 7, 7, 7, 9, 7, 9, 4, 11, 11...",0.600000,0.091275,2.397167,1.416827,0.713183,TRAABXH128F42955D6,Brian Wilson,Keep An Eye On Summer (Album Version)
4,12360,55.062136,11.614828,62,0.561734,0.645350,88.016019,19.987963,3,12.666667,...,"[[0.005479452054779508, 0.09863013698603115, 0...","[2, 2, 2, 9, 9, 2, 2, 2, 4, 2, 2, 2, 9, 2, 11, 2]",0.533333,0.128428,2.555363,1.223556,0.525736,TRAACQE12903CC706C,Old Man River,Summer


In [28]:
features_df.columns

Index(['note_count', 'mean_pitch', 'std_pitch', 'pitch_range', 'mean_duration',
       'std_duration', 'mean_velocity', 'std_velocity', 'num_files',
       'mean_instruments', 'low_chroma', 'mid_chroma', 'high_chroma',
       'total_chroma', 'key', 'key_strength', 'melody_intervals',
       'melody_ngrams', 'low_intervals', 'low_ngrams', 'mid_intervals',
       'mid_ngrams', 'high_intervals', 'high_ngrams', 'ioi_hist',
       'duration_hist', 'density', 'drum_ioi_hist', 'drum_density',
       'bass_ioi_hist', 'bass_density', 'melody_ioi_hist', 'melody_density',
       'segment_chroma', 'chord_progression', 'harmonic_rhythm',
       'segment_variation', 'pitch_entropy', 'duration_entropy', 'ioi_entropy',
       'track_id', 'artist', 'title'],
      dtype='str')

In [17]:
from collections import Counter

global_vocab = Counter()

for ngrams in features_df["melody_ngrams"]:
    global_vocab.update(ngrams)

In [18]:
TOP_NGRAMS = 500

ngram_vocab = {
    ngram: i
    for i, (ngram, _)
    in enumerate(global_vocab.most_common(TOP_NGRAMS))
}

In [19]:
def vectorise_counter(counter, vocab):

    vec = np.zeros(len(vocab))

    total = sum(counter.values())

    if total == 0:
        return vec
    for item, count in counter.items():

        if item in vocab:
            vec[vocab[item]] = count / total

    return vec

In [20]:
def build_embedding(row, ngram_vocab):

    parts = []

    parts.append(row["total_chroma"])
    parts.append(row["low_chroma"])
    parts.append(row["mid_chroma"])
    parts.append(row["high_chroma"])

    parts.append(row["ioi_hist"])
    parts.append(row["duration_hist"])

    parts.append(np.array([
        row["density"],
        row["drum_density"],
        row["bass_density"],
        row["melody_density"]
    ]))

    parts.append(np.array([
        row["pitch_entropy"],
        row["duration_entropy"],
        row["ioi_entropy"]
    ]))

    parts.append(
        vectorise_counter(
            row["melody_ngrams"],
            ngram_vocab
        )
    )

    return np.concatenate(parts)

In [21]:
X = np.vstack([
    build_embedding(row, ngram_vocab)
    for _, row in features_df.iterrows()
])

In [22]:
metadata = features_df[
    [
        "track_id",
        "artist",
        "title",
    ]
].copy()

In [23]:
feature_cols = [
    c for c in features_df.columns
    if c not in [
        "track_id",
        "artist",
        "title",
        "path"
    ]
]

## Light EDA

In [24]:
X.shape

(10, 575)

In [25]:
metadata.head()

,track_id,artist,title
0,TRAAAGR128F425B14B,Cyndi Lauper,Into The Nightlife
1,TRAAAZF12903CCCF6B,Matthew Wilder,Break My Stride
2,TRAABVM128F92CA9DC,Tesla,Caught In A Dream
3,TRAABXH128F42955D6,Brian Wilson,Keep An Eye On Summer (Album Version)
4,TRAACQE12903CC706C,Old Man River,Summer


### PCA

In [26]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [27]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "track": metadata["title"],
    "artist": metadata["artist"]
})

fig = px.scatter(
    df,
    x="PC1",
    y="PC2",
    hover_data=["track", "artist"]
)

fig.show()